# STEP 1: Mount Drive & Install Dependencies

In [ ]:
# STEP 1: Mount Drive & Install Dependencies
from google.colab import drive
import os

drive.mount('/content/drive')
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-geometric

# STEP 2: Imports

In [ ]:
# STEP 2: Imports
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# STEP 3: Paths

In [ ]:
# STEP 3: Paths
data_root          = "path"
train_dir          = os.path.join(data_root, "path")
test_dir           = os.path.join(data_root, "path")
metadata_path      = os.path.join(data_root, "path")
test_metadata_path = os.path.join(data_root, "path")

# STEP 4: Utility to extract upper triangle

In [ ]:
# STEP 4: Utility to extract upper triangle
def extract_upper_triangle(file_path):
    mat = pd.read_csv(file_path, sep='\t', header=None).values
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

# STEP 5: Load and merge raw data

In [ ]:
# STEP 5: Load and merge training data
print("🔄 Loading training data and metadata…")
train_vectors, train_ids = [], []
for fn in tqdm(os.listdir(train_dir)):
    if fn.endswith('.tsv'):
        pid = fn.split('_')[0].replace('sub-','').upper().strip()
        train_vectors.append(extract_upper_triangle(os.path.join(train_dir, fn)))
        train_ids.append(pid)

X_raw = pd.DataFrame(train_vectors)
X_raw['participant_id'] = train_ids

meta = pd.read_csv(metadata_path)
meta['participant_id'] = meta['participant_id'].str.upper().str.strip()

train_df = pd.merge(X_raw, meta, on='participant_id')
print("✅ Training samples:", train_df.shape)

# STEP 6: Bin ages to nearest 0.5

In [ ]:
# STEP 6: Bin ages to nearest 0.5
bins_05 = np.arange(5.0, 21.1, 0.5)
train_df['age_binned'] = (train_df['age'] * 2).round() / 2

# STEP 7: Balance ~65 samples per 0.5-year bin 5.0–21.0

In [ ]:
# STEP 7: Balance ~65 samples per 0.5-year bin
target_n = 65
balanced = []
grouped  = train_df.groupby('age_binned')
for b in bins_05:
    grp = grouped.get_group(b) if b in grouped.groups else pd.DataFrame()
    if grp.empty:
        continue
    if len(grp) >= target_n:
        samp = grp.sample(n=target_n, random_state=42)
    else:
        samp = resample(grp, replace=True, n_samples=target_n, random_state=42)
    balanced.append(samp)

balanced_df = pd.concat(balanced).reset_index(drop=True)
print("✅ Balanced DataFrame:", balanced_df.shape)

# STEP 8: Feature engineering

In [ ]:
# STEP 8: Feature engineering
brain_cols = [c for c in balanced_df.columns if isinstance(c,int) or (isinstance(c,str) and c.isdigit())]
meta_cols  = [c for c in balanced_df.columns if c not in brain_cols + ['participant_id','age','age_binned']]

scaler    = StandardScaler()
brain_sc  = scaler.fit_transform(balanced_df[brain_cols])
pca       = PCA(n_components=100)
brain_pc  = pca.fit_transform(brain_sc)

meta_enc  = pd.get_dummies(balanced_df[meta_cols])
# ensure same columns when reindexing
meta_enc  = meta_enc.reindex(columns=meta_enc.columns, fill_value=0)

# assemble X,y
X         = np.hstack([brain_pc, meta_enc.values])
y         = balanced_df['age'].values
y_scaler  = QuantileTransformer(output_distribution='normal')
y_t       = y_scaler.fit_transform(y.reshape(-1,1)).ravel()

# STEP 9: Train/Val/Test split

In [ ]:
# STEP 9: Train/Val/Test split
# 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y_t, test_size=0.3, random_state=42)
X_val,   X_test, y_val,   y_test   = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# STEP 10: Impute missing

In [ ]:
# STEP 10: Imputation
imp = SimpleImputer(strategy='mean')
X_train = imp.fit_transform(X_train)
X_val   = imp.transform(X_val)
X_test  = imp.transform(X_test)

# STEP 11: Train XGBoost with balanced sample weights

In [ ]:
# STEP 11: Train XGBoost with sample weights
age_freq = balanced_df['age'].value_counts(normalize=True)
weights  = balanced_df['age'].map(lambda a:1/age_freq[a]).values[:len(X_train)]

model = XGBRegressor(
    n_estimators     = 300,
    learning_rate   = 0.05,
    max_depth       = 8,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    random_state    = 42
)
model.fit(X_train, y_train, sample_weight=weights)
print("✅ Model trained.")

# STEP 12: Evaluate

In [ ]:
# STEP 12: Evaluation
def inv(arr): return y_scaler.inverse_transform(arr.reshape(-1,1)).ravel()
def metrics(name, yt, yp):
    mse  = mean_squared_error(yt, yp)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(yt, yp)
    print(f"{name:5s} | R2: {r2_score(yt,yp):.3f}, RMSE: {rmse:.3f}, MAE: {mae:.3f}")

for split,Xs,ys in zip(["Train","Val","Test"], [X_train,X_val,X_test], [y_train,y_val,y_test]):
    preds = inv(model.predict(Xs))
    metrics(split, inv(ys), preds)

# STEP 13: Prepare submission

In [ ]:
# STEP 13: Prepare submission
print("\n🔄 Preparing submission data…")
tst_vecs, tst_ids = [], []
for fn in tqdm(os.listdir(test_dir)):
    if fn.endswith('.tsv'):
        pid = fn.split('_')[0].replace('sub-','').upper().strip()
        tst_vecs.append(extract_upper_triangle(os.path.join(test_dir, fn)))
        tst_ids.append(pid)

X_traw = pd.DataFrame(tst_vecs)
X_traw['participant_id'] = tst_ids

# 13a) For feature construction, merge with test_metadata.csv (no 'age')
tmeta     = pd.read_csv(test_metadata_path)
tmeta['participant_id'] = tmeta['participant_id'].str.upper().str.strip()
test_df   = pd.merge(X_traw, tmeta, on='participant_id')
print("✅ Test features:", test_df.shape)

# 13b) To verify true ages present, merge X_traw with training_metadata.csv
all_meta = pd.read_csv(metadata_path)
all_meta['participant_id'] = all_meta['participant_id'].str.upper().str.strip()
temp     = pd.merge(X_traw, all_meta[['participant_id','age']], on='participant_id')
print("True ages in test set:", sorted(temp['age'].unique()))

# STEP 14: Features for submission

In [ ]:
# STEP 14: Feature build & predict
br_test_sc = scaler.transform(test_df[brain_cols])
br_test_pc = pca.transform(br_test_sc)
md_test_enc= pd.get_dummies(test_df[meta_cols]).reindex(columns=meta_enc.columns, fill_value=0)
X_submit   = np.hstack([br_test_pc, md_test_enc.values])
X_submit   = imp.transform(X_submit)

preds      = inv(model.predict(X_submit))
preds      = np.clip(preds, 5, 21)

submission = pd.DataFrame({
    'participant_id': test_df['participant_id'],
    'age'           : preds
})
submission.to_csv(os.path.join(data_root,'balanced_submission.csv'), index=False)
print("✅ Saved balanced_submission.csv")

✅ Saved balanced_submission.csv


# STEP 15: Visualize distribution

In [ ]:
# STEP 15: Visualize distribution
sns.histplot(preds, bins=len(bins_05), kde=False)
plt.title('Predicted Age Distribution (0.5-year bins)')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
import pandas as pd

# Load the training metadata
metadata_path = "path"
df = pd.read_csv(metadata_path)

# Convert float ages to integer by flooring (or use round if preferred)
df["age_int"] = df["age"].astype(int)

# Count the number of samples per integer age
age_distribution = df["age_int"].value_counts().sort_index().reset_index()
age_distribution.columns = ["age", "count"]

# Add percentage column
age_distribution["percentage"] = 100 * age_distribution["count"] / age_distribution["count"].sum()

# Display the results
print(age_distribution)